# relu-elementwise-max — worked example 2: Compute the ReLU Derivative Mask Manually

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `relu-elementwise-max`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The derivative of ReLU is 1 for positive inputs and 0 for negative inputs (it is undefined at exactly 0, but the sub-gradient 0 is typically used). During backpropagation, the ReLU backward pass multiplies the incoming gradient by this binary mask. Understanding this mask helps explain why dying ReLUs occur: neurons with permanently negative pre-activations stop passing gradients entirely.

## Worked solution

**Step 1 — define the derivative rule.** `relu'(x) = 1` if `x > 0`, else `0`. In PyTorch, this is `(x > 0).float()`.

**Step 2 — compute `relu_grad(grad_out, x)`.** This function takes the upstream gradient `grad_out` and the forward input `x`, and returns `grad_out * (x > 0).float()`. Negative-input positions get zero gradient; positive-input positions pass gradient through unchanged.

**Step 3 — manually check with autograd.** Apply `t.maximum(x, 0)`, sum the output, and call `.backward()`. The `.grad` on `x` should match our manual mask.

**Step 4 — print and compare.** Show both the autograd gradient and the manually computed mask side-by-side.

**Step 5 — note the at-zero behavior.** PyTorch's `t.maximum` gives sub-gradient 0.5 at exactly `x=0`, while our mask gives 0. Both are valid; the manual mask is the more common convention.

In [ ]:
import torch as t

def relu_grad(grad_out: t.Tensor, x: t.Tensor) -> t.Tensor:
    """Backward pass for ReLU: pass gradient only where x > 0."""
    mask = (x > 0).float()
    return grad_out * mask

# --- exercise and print ---
t.manual_seed(5)
x_vals = t.tensor([-2.0, -0.1, 1.0, 3.5, -1.0, 2.0])

# Manual backward
grad_out = t.ones_like(x_vals)
manual_grad = relu_grad(grad_out, x_vals)

# Autograd backward (for comparison)
x_ag = x_vals.clone().requires_grad_(True)
y = t.maximum(x_ag, t.tensor(0.0))
y.sum().backward()

print('x:           ', x_vals.tolist())
print('manual grad: ', manual_grad.tolist())   # 0s and 1s
print('autograd:    ', x_ag.grad.tolist())     # same (except 0.5 at x=0 positions)
print('Match (non-zero x):', t.allclose(
    manual_grad[x_vals != 0], x_ag.grad[x_vals != 0]
))